# Fedlex fine-tuning on a free Kaggle GPU

Fine-tunes `intfloat/multilingual-e5-small` for cross-lingual retrieval over Swiss federal law with `fedlex_bench.finetune.train` (plain PyTorch, in-batch ranking loss). Background: `docs/learning/04_loss_and_training_knobs.md`.

**Before you run (once):**
1. Upload `fedlex_finetune_kit.zip` as a **private** Kaggle dataset and add it to this notebook (right panel, *Add Input*).
2. Settings (right panel): **Accelerator = GPU T4 x2 or P100**, **Internet = On** (needed to download the model weights from Hugging Face; Kaggle asks for phone verification once).
3. Set `LEARNING_RATE` and `EPOCHS` in the next cell. They are your decisions: write your prediction in `docs/learning/journal.md` first.

Then *Run All*. Total time on a T4 should be minutes per run. Download the contents of `/kaggle/working/out` from the *Output* tab when it is done.

In [ ]:
# ---- Settings: YOUR decisions ------------------------------------------------------------------
LEARNING_RATE = None   # TODO(human): typical for a small encoder: 1e-5 to 5e-5
EPOCHS = None          # TODO(human): 1 to 5; more epochs risk overfitting 11.5k pairs

BATCH_SIZE = 64        # also the number of in-batch negatives (batch - 1)
RUN_ABLATION = True    # data-size runs (25% and 50% of training articles) after the two main runs
ABLATION_HARD_NEGATIVES = False   # set True if run 2 (hard negatives) beats run 1 on validation
# -------------------------------------------------------------------------------------------------
assert LEARNING_RATE and EPOCHS, "Set LEARNING_RATE and EPOCHS in this cell first."

In [ ]:
import glob, os, pathlib, shutil, subprocess, sys, zipfile

WORK = pathlib.Path("/kaggle/working")
ROOT = WORK / "fedlex"
hits = glob.glob("/kaggle/input/**/src/fedlex_bench", recursive=True)
zips = glob.glob("/kaggle/input/**/fedlex_finetune_kit.zip", recursive=True)
if ROOT.exists():
    shutil.rmtree(ROOT)
if hits:                                  # Kaggle unpacked the zip into the dataset folder
    shutil.copytree(pathlib.Path(hits[0]).parents[1], ROOT)
elif zips:                                # the zip arrived as a file
    zipfile.ZipFile(zips[0]).extractall(ROOT)
else:
    raise FileNotFoundError("Add the fedlex_finetune_kit dataset as an input of this notebook.")
print("kit at", ROOT, "| git sha:", __import__("json").load(open(ROOT / "KIT_INFO.json"))["git_sha"][:8])

In [ ]:
%pip install -q python-dotenv lxml httpx pytest
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "Turn on a GPU: right panel > Settings > Accelerator."
print(torch.cuda.get_device_name(0))

In [ ]:
# Gate: the leakage tests and the batching/loss tests must pass before any training.
env = {**os.environ, "PYTHONPATH": str(ROOT / "src")}
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider",
                    "tests/test_finetune_leakage.py", "tests/test_finetune_train.py"],
                   cwd=ROOT, env=env, capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-500:])
assert r.returncode == 0, "Tests failed: do not train."

In [ ]:
def run(name, extra=()):
    cmd = [sys.executable, "-u", "-m", "fedlex_bench.finetune.train", "--run", name,
           "--lr", str(LEARNING_RATE), "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE),
           "--amp", *extra]
    p = subprocess.Popen(cmd, cwd=ROOT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if "Loading weights" not in line:
            print(line, end="")
    assert p.wait() == 0, f"{name} failed"

# Speed check: 20 steps, saves nothing. Prints seconds per step on this GPU.
run("timing", ["--max-steps", "20"])

## Run 1: in-batch negatives only

In [ ]:
run("r1_inbatch")

## Run 2: in-batch plus one hard negative per question

In [ ]:
run("r2_hardneg", ["--hard-negatives"])

## Learning curves

In [ ]:
import json
import matplotlib.pyplot as plt

def load(name):
    return [json.loads(l) for l in open(ROOT / "models" / name / "log.jsonl")]

runs = [n for n in ("r1_inbatch", "r2_hardneg") if (ROOT / "models" / n / "log.jsonl").exists()]
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for n in runs:
    log = load(n)
    ep = [r["epoch"] for r in log]
    ax[0].plot([r["epoch"] for r in log if r["train_loss"] is not None],
               [r["train_loss"] for r in log if r["train_loss"] is not None], "o-", label=f"{n} train")
    ax[0].plot(ep, [r["val_loss"] for r in log], "s--", label=f"{n} validation")
    ax[1].plot(ep, [r["val_r10_all"] for r in log], "o-", label=n)
    ax[2].plot(ep, [r["val_r10_en"] for r in log], "o-", label=f"{n} English")
    ax[2].plot(ep, [r["val_r10_de"] for r in log], "x:", label=f"{n} German")
ax[0].set_title("loss per epoch (epoch 0 = untrained)"); ax[1].set_title("validation recall@10 (324 articles)")
ax[2].set_title("validation recall@10: English vs German")
for a in ax: a.set_xlabel("epoch"); a.legend(fontsize=7); a.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(WORK / "learning_curves.png", dpi=130); plt.show()
for n in runs:
    print(n); [print("  ", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in r.items()
                          if k in ("epoch", "train_loss", "val_loss", "val_r1_all", "val_r10_all", "val_r10_en", "val_r10_de", "val_r10_fr", "val_r10_it")}) for r in load(n)]

## Data-size ablation (25% and 50% of the training articles; 100% is run 1 or 2)

In [ ]:
if RUN_ABLATION:
    for frac in (0.25, 0.5):
        extra = ["--fraction", str(frac)] + (["--hard-negatives"] if ABLATION_HARD_NEGATIVES else [])
        run(f"r3_frac{int(frac * 100)}", extra)

## Package the results

In [ ]:
OUT = WORK / "out"
OUT.mkdir(exist_ok=True)
for d in sorted((ROOT / "models").iterdir()):
    if not (d / "config.json").exists():
        continue
    for f in ("log.jsonl", "config.json"):
        shutil.copy(d / f, OUT / f"{d.name}_{f}")
    shutil.make_archive(str(OUT / f"{d.name}_best"), "zip", d / "best")
if (WORK / "learning_curves.png").exists():
    shutil.copy(WORK / "learning_curves.png", OUT)
for f in sorted(OUT.iterdir()):
    print(f"{f.stat().st_size / 1e6:8.1f} MB  {f.name}")